# Relative 因子：按规则筛选 + 按相关方向等权融合

1. 用 **config** 中关于 **lag** 与 **名字** 的规则筛选显著 relative 因子
2. 按各因子与 y 的 **相关系数方向**（正/负）决定融合时的符号，做 **等权融合**

**依赖**：请先运行 **01、02、03**（或本 notebook 将自行重建 df_y、relative_factors 与回归结果）。

In [1]:
import os
import sys
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

_cwd = os.getcwd()
_root = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'factor_build' else _cwd
if _root not in sys.path:
    sys.path.insert(0, _root)
import config

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

DATA_DIR = config.get_data_dir()
DATE_COL = config.DATE_COL
PCT_COL = config.PCT_COL

LAG_DAYS = getattr(config, 'REGRESSION_LAG_DAYS', [0, 1, 2, 3])
MAX_P = getattr(config, 'REGRESSION_MAX_PVALUE', 0.05)
FUSION_LAG = getattr(config, 'FUSION_LAG_ALLOWED', [0, 1, 2, 3])
FUSION_SHEET = getattr(config, 'FUSION_SHEET_INCLUDE', None)
FUSION_PAIR = getattr(config, 'FUSION_PAIR_INCLUDE', None)
ROLLING_ZSCORE_WINDOW = getattr(config, 'ROLLING_ZSCORE_WINDOW', 60)
ROLLING_ZSCORE_MIN_VALID_IN_WINDOW = getattr(config, 'ROLLING_ZSCORE_MIN_VALID_IN_WINDOW', 60)

print("ROLLING_ZSCORE_WINDOW:",ROLLING_ZSCORE_WINDOW)
print("ROLLING_ZSCORE_MIN_VALID_IN_WINDOW:",ROLLING_ZSCORE_MIN_VALID_IN_WINDOW)


ROLLING_ZSCORE_WINDOW: 90
ROLLING_ZSCORE_MIN_VALID_IN_WINDOW: 90


## 1. 准备 df_y、relative_factors、回归结果 reg_df

若内存中无则重建（与 03 一致）。

In [2]:
def get_pair_cols(df):
    return [c for c in df.columns if c != DATE_COL]

try:
    df_y
except NameError:
    all_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.xlsx') and not f.startswith(config.SKIP_FILE_PREFIX)]
    file_50 = next((f for f in all_files if all(k in f for k in config.FILE_KEYWORDS_CSI50)), None)
    file_1000 = next((f for f in all_files if all(k in f for k in config.FILE_KEYWORDS_CSI1000)), None)
    df50 = pd.read_excel(os.path.join(DATA_DIR, file_50))[[DATE_COL, PCT_COL]].dropna()
    df1000 = pd.read_excel(os.path.join(DATA_DIR, file_1000))[[DATE_COL, PCT_COL]].dropna()
    df50[DATE_COL] = pd.to_datetime(df50[DATE_COL]).dt.date
    df1000[DATE_COL] = pd.to_datetime(df1000[DATE_COL]).dt.date
    df50 = df50.rename(columns={PCT_COL: '涨跌幅_50'})
    df1000 = df1000.rename(columns={PCT_COL: '涨跌幅_1000'})
    df_y = df50.merge(df1000, on=DATE_COL, how='inner')
    df_y = df_y.dropna()
    df_y['y'] = df_y['涨跌幅_50'].astype(float) - df_y['涨跌幅_1000'].astype(float)
    df_y = df_y[[DATE_COL, 'y']].reset_index(drop=True)
    if getattr(config, 'CUTOFF_DATE', None):
        import datetime
        cut = config.CUTOFF_DATE
        if isinstance(cut, (tuple, list)) and len(cut) >= 3:
            cut = datetime.date(int(cut[0]), int(cut[1]), int(cut[2]))
        df_y = df_y[df_y[DATE_COL] <= cut]
try:
    relative_factors
    sheet_names
except NameError:
    factor_path = os.path.join(DATA_DIR, getattr(config, 'AUGMENTED_FACTOR_FILENAME', '因子_国家队_增强版_pair_zscore.xlsx'))
    sheets_raw = pd.read_excel(factor_path, sheet_name=None)
    EXCLUDE = getattr(config, 'EXCLUDE_SHEETS', [])
    sheet_names = [n for n in sheets_raw.keys() if n != 'Target' and n not in EXCLUDE]
    ROLLING_WINDOW = getattr(config, 'ROLLING_ZSCORE_WINDOW', 60)
    MIN_VALID = getattr(config, 'ROLLING_ZSCORE_MIN_VALID_IN_WINDOW', 60)
    RELATIVE_PAIRS = getattr(config, 'RELATIVE_FACTOR_PAIRS', [])
    print("ROLLING_WINDOW:",ROLLING_WINDOW)
    print("MIN_VALID:",MIN_VALID)
    print("RELATIVE_PAIRS:",RELATIVE_PAIRS)
    def normalize_date(df):
        d = df.copy()
        if 'Date' in d.columns and DATE_COL not in d.columns:
            d = d.rename(columns={'Date': DATE_COL})
        if DATE_COL in d.columns:
            d[DATE_COL] = pd.to_datetime(d[DATE_COL], errors='coerce').dt.date
        return d
    def rolling_zscore(series, window):
        x = pd.to_numeric(series, errors='coerce')
        valid = x.notna()
        x_for_roll = x.where(valid)
        roll_mean = x_for_roll.rolling(window=window, min_periods=MIN_VALID).mean()
        roll_std = x_for_roll.rolling(window=window, min_periods=MIN_VALID).std()
        z = (x - roll_mean) / roll_std.where(roll_std > 1e-12)
        return z.where(valid)
    augmented_sheets = {}
    for name in sheet_names:
        df = normalize_date(sheets_raw[name].copy())
        for col in get_pair_cols(df):
            if col in df.columns:
                raw = pd.to_numeric(df[col], errors='coerce').ffill()
                df[col] = rolling_zscore(raw, ROLLING_WINDOW)
        augmented_sheets[name] = df
    relative_factors = {}
    for name in sheet_names:
        df_z = augmented_sheets[name]
        out = df_z[[DATE_COL]].copy()
        for code_a, code_b in RELATIVE_PAIRS:
            if code_a in df_z.columns and code_b in df_z.columns:
                out[f"{code_a}-{code_b}"] = df_z[code_a] - df_z[code_b]
        relative_factors[name] = out

df_y_sorted = df_y.sort_values(DATE_COL).reset_index(drop=True)
try:
    reg_df
except NameError:
    INTRADAY_SHEETS = getattr(config, 'LAG_INTRADAY_SHEETS', [])
    INTRADAY_DAYS   = getattr(config, 'LAG_INTRADAY_DAYS',   [0])
    DELAYED_SHEETS  = getattr(config, 'LAG_DELAYED_SHEETS',  [])
    DELAYED_DAYS    = getattr(config, 'LAG_DELAYED_DAYS',    [-3, -2, -1])
    reg_results = []
    for name in sheet_names:
        df_f = relative_factors[name]
        if name in INTRADAY_SHEETS:
            lags = INTRADAY_DAYS
        elif name in DELAYED_SHEETS:
            lags = DELAYED_DAYS
        else:
            lags = LAG_DAYS
        for col in get_pair_cols(df_f):
            for lag in lags:
                df_yt = df_y_sorted[[DATE_COL]].copy()
                df_yt['y_target'] = df_y_sorted['y'].shift(-lag)
                m = df_yt.merge(df_f[[DATE_COL, col]], on=DATE_COL, how='inner').dropna()
                if len(m) < 10:
                    reg_results.append({'sheet': name, '标的对': col, 'lag': lag, 'n': len(m), 'corr': np.nan, 'p_value': np.nan})
                    continue
                y, x = m['y_target'].astype(float), m[col].astype(float)
                r, p = pearsonr(y, x)
                reg_results.append({'sheet': name, '标的对': col, 'lag': lag, 'n': len(m), 'corr': round(r, 4), 'p_value': p})
    reg_df = pd.DataFrame(reg_results)
    reg_df['abs_corr'] = reg_df['corr'].abs()

sig = reg_df[reg_df['p_value'] < MAX_P].copy()
print('显著因子数（p < {}）:'.format(MAX_P), len(sig))
print('reg_df 行数:', len(reg_df))

ROLLING_WINDOW: 90
MIN_VALID: 90
RELATIVE_PAIRS: [('510050.SH', '510100.SH'), ('588080.SH', '588050.SH'), ('159915.SZ', '159952.SZ'), ('159915.SZ', '159977.SZ')]
显著因子数（p < 0.05）: 8
reg_df 行数: 24


## 2. 按 config 规则筛选：lag 与名字

In [3]:
# 新融合规则：每种因子（sheet）只取 |corr| 最大的那一个 (标的对, lag) 组合
sig['abs_corr'] = sig['corr'].abs()
best_idx = sig.groupby('sheet')['abs_corr'].idxmax()
filtered = sig.loc[best_idx].copy()
if FUSION_SHEET is not None and len(FUSION_SHEET) > 0:
    filtered = filtered[filtered['sheet'].isin(FUSION_SHEET)]
if FUSION_PAIR is not None and len(FUSION_PAIR) > 0:
    mask = filtered['标的对'].apply(lambda s: any(p in s for p in FUSION_PAIR))
    filtered = filtered[mask]
filtered = filtered.reset_index(drop=True)

print('融合规则: 每个因子(sheet)取 |corr| 最强的 (标的对, lag) 组合')
print('         FUSION_SHEET_INCLUDE =', FUSION_SHEET)
print('         FUSION_PAIR_INCLUDE =', FUSION_PAIR)
print('筛选后因子数:', len(filtered))
display(filtered)

融合规则: 每个因子(sheet)取 |corr| 最强的 (标的对, lag) 组合
         FUSION_SHEET_INCLUDE = None
         FUSION_PAIR_INCLUDE = None
筛选后因子数: 4


,sheet,标的对,lag,n,corr,p_value,abs_corr
0,NAV_iopv_discount,510050.SH-510100.SH,0,1392,0.2049,1.171520e-14,0.2049
1,netinflow,159915.SZ-159952.SZ,-2,1576,0.0582,2.076305e-02,0.0582
2,turn,588080.SH-588050.SH,0,1123,0.1233,3.422549e-05,0.1233
3,volume_btin,588080.SH-588050.SH,0,1123,0.1143,1.240283e-04,0.1143


In [4]:
filtered

,sheet,标的对,lag,n,corr,p_value,abs_corr
0,NAV_iopv_discount,510050.SH-510100.SH,0,1392,0.2049,1.171520e-14,0.2049
1,netinflow,159915.SZ-159952.SZ,-2,1576,0.0582,2.076305e-02,0.0582
2,turn,588080.SH-588050.SH,0,1123,0.1233,3.422549e-05,0.1233
3,volume_btin,588080.SH-588050.SH,0,1123,0.1143,1.240283e-04,0.1143


## 3. 按相关方向等权融合（按 lag 对齐到同一预测日）

对每个 (sheet, 标的对) 取 |corr| 最大对应的 **lag** 与 **sign**。融合时按 **目标日 T** 对齐：因子 i 在 T 日的贡献 = sign_i × 因子 i 在 (T − lag_i) 日的值，使融合因子预测的是**同一天**的大盘减小盘。

In [5]:
# 排除 volume, volume_btin, amt, amt_btin, turn 类因子
EXCLUDE_SHEETS = []
filtered = filtered[~filtered['sheet'].isin(EXCLUDE_SHEETS)]

# 只保留 corr != 0 的因子参与融合
filtered_pos = filtered[filtered['corr'] != 0].copy()

# 每个 (sheet, 标的对) 取 |corr| 最大的一行，得到融合用 lag；sign = corr 的符号（正→+1，负→-1）
best = filtered_pos.loc[filtered_pos.groupby(['sheet', '标的对'])['abs_corr'].idxmax()][['sheet', '标的对', 'corr', 'lag']].copy()
best['sign'] = np.where(best['corr'] >= 0, 1.0, -1.0)

# 交易日历：用 y 的日期作为目标日，按位置 +lag 对齐因子
all_dates = df_y_sorted[DATE_COL].drop_duplicates().tolist()
date_to_idx = {d: i for i, d in enumerate(all_dates)}
idx_to_date = {i: d for i, d in enumerate(all_dates)}
n_dates = len(all_dates)

merged = None
for _, row in best.iterrows():
    name, col, sign, lag = row['sheet'], row['标的对'], row['sign'], int(row['lag'])
    df_f = relative_factors[name][[DATE_COL, col]].copy()
    df_f['pos'] = df_f[DATE_COL].map(date_to_idx)
    df_f = df_f.dropna(subset=['pos'])
    df_f['target_pos'] = df_f['pos'].astype(int) + lag
    df_f = df_f[df_f['target_pos'].between(0, n_dates - 1)]
    df_f['target_date'] = df_f['target_pos'].map(idx_to_date)
    contrib = sign * pd.to_numeric(df_f[col], errors='coerce')
    out = df_f[['target_date']].copy()
    out[f'_x_{name}_{col}'] = contrib.values
    if merged is None:
        merged = out
    else:
        merged = merged.merge(out, on='target_date', how='outer')

date_col = DATE_COL
merged = merged.rename(columns={'target_date': date_col})
merged = merged.sort_values(date_col).reset_index(drop=True)  # 按日期排序，rolling 才有意义
x_cols = [c for c in merged.columns if c != date_col]
if len(x_cols) == 0:
    raise ValueError('没有可融合的因子序列')

# 融合前标准化：对每列做 rolling z-score，窗口与 config 统一
_WINDOW = ROLLING_ZSCORE_WINDOW
_MIN_VALID = ROLLING_ZSCORE_MIN_VALID_IN_WINDOW

def _rolling_zscore(series, window, min_valid):
    """Rolling z-score：与 02 中对原始因子的标准化方式一致。"""
    x = pd.to_numeric(series, errors='coerce')
    valid = x.notna()
    x_for_roll = x.where(valid)
    roll_mean = x_for_roll.rolling(window=window, min_periods=min_valid).mean()
    roll_std  = x_for_roll.rolling(window=window, min_periods=min_valid).std()
    z = (x - roll_mean) / roll_std.where(roll_std > 1e-12)
    return z.where(valid)

for c in x_cols:
    merged[c] = _rolling_zscore(merged[c], _WINDOW, _MIN_VALID)

merged['fusion'] = merged[x_cols].mean(axis=1)
df_fusion = merged[[date_col, 'fusion']].dropna(subset=['fusion']).sort_values(date_col).reset_index(drop=True)

print('融合前 rolling z-score 窗口:', _WINDOW, '，最小有效值:', _MIN_VALID)
print('参与融合的因子:')
display(best)
print('融合序列长度:', len(df_fusion))
display(df_fusion.head(10))

# 输出融合因子时间序列到 factor_build/outputs（每次运行覆盖更新）
_out_dir = os.path.join(os.getcwd(), getattr(config, 'FACTOR_BUILD_OUTPUTS', 'outputs'))
os.makedirs(_out_dir, exist_ok=True)
_path_fusion = os.path.join(_out_dir, '04_fusion_timeseries.xlsx')
df_fusion.to_excel(_path_fusion, index=False)
print('已输出:', _path_fusion)
# 筛选出的、用于构成 fusion 的 relative 因子对（sheet, 标的对, corr, sign）
_path_const = os.path.join(_out_dir, '04_fusion_constituents.xlsx')
best.to_excel(_path_const, index=False)
print('已输出:', _path_const)

融合前 rolling z-score 窗口: 90 ，最小有效值: 90
参与融合的因子:


,sheet,标的对,corr,lag,sign
0,NAV_iopv_discount,510050.SH-510100.SH,0.2049,0,1.0
1,netinflow,159915.SZ-159952.SZ,0.0582,-2,1.0
2,turn,588080.SH-588050.SH,0.1233,0,1.0
3,volume_btin,588080.SH-588050.SH,0.1143,0,1.0


融合序列长度: 1489


,交易日期,fusion
0,2019-09-20,0.143279
1,2019-09-23,-0.843299
2,2019-09-24,-0.532238
3,2019-09-25,1.978535
4,2019-09-26,-0.940186
5,2019-09-27,0.261302
6,2019-09-30,0.324842
7,2019-10-08,1.552454
8,2019-10-09,-0.936783
9,2019-10-10,-1.121589


已输出: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_build/outputs/04_fusion_timeseries.xlsx
已输出: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_build/outputs/04_fusion_constituents.xlsx


## 4. 汇总

- **filtered**：按 lag/名字规则筛选后的显著因子表  
- **best**：每个 (sheet, 标的对) 取 |corr| 最大对应的 sign  
- **df_fusion**：等权融合后的时序（交易日期, fusion）

In [6]:
print('df_fusion 日期范围:', df_fusion[DATE_COL].min(), '~', df_fusion[DATE_COL].max())
print('fusion 统计: mean = {:.4f}, std = {:.4f}'.format(df_fusion['fusion'].mean(), df_fusion['fusion'].std()))

df_fusion 日期范围: 2019-09-20 ~ 2025-11-13
fusion 统计: mean = 0.0124, std = 0.7696
